In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

In [2]:
# MLP-VQVAE training
#Import custom classes and libraries
from src.data.data_loading import L1TriggerDataset
from src.data.data_loading import L1TriggerDataModule
from src.models.mlp_vqvae import MLPVQVAE

import yaml
from tqdm import tqdm
import lightning as pl
import torch
import matplotlib.pyplot as plt


#Open config files
with open("../configs/data.yaml") as f:
    data_config = yaml.safe_load(f)

#Open config file
with open("../configs/model.yaml") as f:
    model_config = yaml.safe_load(f)

#Extract config info
#Directories
dirs_train = data_config["data"]["train_path"]
dirs_val = data_config["data"]["val_path"]
dirs_test = data_config["data"]["test_path"]

#Features
features_cols = data_config["data"]["features"]
max_part = data_config["data"]["max_particles"]
prep = data_config["data"]["preprocessing"]

#VQVAE hyperparameters
input_dim = model_config["MLP_VQVAE"]["input_dim"]
hidden_dim = model_config["MLP_VQVAE"]["hidden_dim"]
latent_dim = model_config["MLP_VQVAE"]["latent_dim"]
codebook_size = model_config["MLP_VQVAE"]["codebook_size"]
rot_trick = model_config["MLP_VQVAE"]["rotation_trick"]
decay = model_config["MLP_VQVAE"]["decay"]
beta = model_config["MLP_VQVAE"]["beta"]

#Training hyperparameters
lr = model_config["training"]["lr"]
max_epochs = model_config["training"]["max_epochs"]
batch_size = model_config["training"]["batch_size"]

In [3]:
#Initialization of the lightining datamodule
data_module = L1TriggerDataModule(
    parquet_dirs_train=dirs_train, 
    parquet_dirs_val=dirs_val,
    parquet_dirs_test=dirs_test,
    max_particles=max_part,
    batch_size=batch_size,
    #num_workers=7,
    features=features_cols,
    preprocessing=prep    
)

In [4]:
#Model
model = MLPVQVAE(
    input_dim=input_dim,
    hidden_dims=hidden_dim,
    decay=decay,
    embedding_dim=latent_dim,
    num_embeddings=codebook_size,
    rot_trick=rot_trick,
    commitment_cost = beta,
    #reconstruction_weight = 1.0
    lr=lr
)

#Trainer (Lightning)
trainer = pl.Trainer(
    max_epochs=2,
    accelerator="auto",   #CPU/GPU automatic
    devices="auto",
    log_every_n_steps=10
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/home/francesco/miniconda3/envs/tesi/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [5]:
#Train
trainer.fit(model, datamodule=data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name      | Type           | Params | Mode  | FLOPs
-------------------------------------------------------------
0 | encoder   | MLPEncoder     | 41.6 K | train | 0    
1 | quantizer | VectorQuantize | 0      | train | 0    
2 | decoder   | MLPDecoder     | 41.3 K | train | 0    
-------------------------------------------------------------
82.9 K    Trainable params
0         Non-trainable params
82.9 K    Total params
0.332     Total estimated model params size (MB)
18        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/francesco/miniconda3/envs/tesi/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/francesco/miniconda3/envs/tesi/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


/home/francesco/miniconda3/envs/tesi/lib/python3.13/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/francesco/miniconda3/envs/tesi/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Epoch 1: |          | 542/? [00:23<00:00, 23.11it/s, v_num=13, train_loss=0.0326, recon_loss=0.0325, commit_loss=0.000131, val_loss=0.0334, val_recon_loss=0.0334, val_commit_loss=0.000]

`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: |          | 542/? [00:23<00:00, 23.10it/s, v_num=13, train_loss=0.0326, recon_loss=0.0325, commit_loss=0.000131, val_loss=0.0334, val_recon_loss=0.0334, val_commit_loss=0.000]
